# LLM Phase — Notebook 2: Provider Report Generator



## Section 1: Setup & Imports

In [ ]:
import os
import json
import pickle
import warnings
from pathlib import Path
from typing import List, Dict, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import shap
import faiss
from dotenv import load_dotenv
from anthropic import Anthropic
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

warnings.filterwarnings('ignore')
print('All packages imported.')



## Section 2: Configuration

In [ ]:
ENV_FILE = Path('/Users/roya/Desktop/Thesis/Thesis_Project/LLMs/ATT18160.env')
load_dotenv(ENV_FILE)
client       = Anthropic(api_key=os.environ.get('OPENAI_API_KEY'))  
CLAUDE_MODEL = 'claude-sonnet-4-6'  
EMBED_MODEL  = 'all-MiniLM-L6-v2'           

st_model = SentenceTransformer(EMBED_MODEL)

ML_DIR         = Path('/Users/roya/Desktop/Thesis/Thesis_Project/ML')
LLM_OUTPUT_DIR = Path('/Users/roya/Desktop/Thesis/Thesis_Project/LLMs/outputs')
VECTOR_DB_DIR  = LLM_OUTPUT_DIR / 'vector_db'
REPORTS_DIR    = LLM_OUTPUT_DIR / 'reports'
FIGURES_DIR    = LLM_OUTPUT_DIR / 'figures'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

TARGET_RECALL           = 0.85  
MAX_PROVIDERS_TO_REPORT = 20    
TOP_SHAP_FEATURES       = 5     

print(f'Reports directory:    {REPORTS_DIR}')
print(f'Vector DB directory:  {VECTOR_DB_DIR}')
print(f'Embedding model:      {EMBED_MODEL} (local)')
print(f'Claude model:         {CLAUDE_MODEL}')



## Section 3: Load Vector Database (from Notebook 1)

In [ ]:

def load_vector_database():
    
    index = faiss.read_index(str(VECTOR_DB_DIR / 'medicare_policy.faiss'))
    with open(VECTOR_DB_DIR / 'chunk_texts.pkl', 'rb') as f:
        chunk_texts = pickle.load(f)
    with open(VECTOR_DB_DIR / 'chunk_metadata.pkl', 'rb') as f:
        chunk_metadata = pickle.load(f)
    print(f'Loaded FAISS index: {index.ntotal:,} vectors')
    print(f'Chunk texts: {len(chunk_texts):,} chunks')
    return index, chunk_texts, chunk_metadata


def embed_query(query: str) -> np.ndarray:
    vec = st_model.encode([query], convert_to_numpy=True)[0].astype(np.float32)
    return vec / (np.linalg.norm(vec) + 1e-10)


def retrieve(query: str, index, chunk_texts, chunk_metadata, top_k=5):
    q = embed_query(query).reshape(1, -1)
    scores, indices = index.search(q, top_k)
    results = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
        if idx == -1:
            continue
        meta = chunk_metadata[idx]
        results.append({
            'rank':            rank,
            'score':           float(score),
            'text':            chunk_texts[idx],
            'source_file':     meta['source_file'],
            'source_category': meta['source_category'],
            'start_page':      meta.get('start_page'),
            'end_page':        meta.get('end_page'),
            'chapter':         meta.get('chapter'),
            'subsection':      meta.get('subsection'),
            'chunk_id':        meta.get('chunk_id'),
        })
    return results


rag_index, chunk_texts, chunk_metadata = load_vector_database()



## Section 4: Load ML Data & Trained Model


# 4.1

In [ ]:
X_test  = pd.read_csv(ML_DIR / 'datasets' / 'X_test_reduced.csv', index_col='original_index')
y_test  = pd.read_csv(ML_DIR / 'datasets' / 'y_test.csv', index_col='original_index').squeeze()

X_train = pd.read_csv(ML_DIR / 'datasets' / 'X_train_reduced.csv', index_col='original_index')
y_train = pd.read_csv(ML_DIR / 'datasets' / 'y_train.csv', index_col='original_index').squeeze()

balanced_path_X = ML_DIR / 'datasets' / 'X_train_balanced.csv'
balanced_path_y = ML_DIR / 'datasets' / 'y_train_balanced.csv'
if balanced_path_X.exists():
    X_train_balanced = pd.read_csv(balanced_path_X, index_col='balanced_index')
    y_train_balanced = pd.read_csv(balanced_path_y, index_col='balanced_index').squeeze()
    print(f'X_train_balanced: {X_train_balanced.shape}  (SMOTE-ENN)')
    print(f'y_train_balanced: {y_train_balanced.shape}  fraud rate={y_train_balanced.mean():.2%}')
else:
    X_train_balanced = None
    y_train_balanced = None
    print('WARNING: X_train_balanced.csv not found — re-run ML_Fraud_Detection.ipynb cell 105 to save it.')
    print('         SHAP will fall back to raw X_train (less accurate).')

feature_names = X_test.columns.tolist()
print(f'\nX_test shape:  {X_test.shape}')
print(f'y_test shape:  {y_test.shape}')
print(f'Fraud cases in test set: {y_test.sum()} / {len(y_test)}')
print(f'Features: {len(feature_names)}')


# 4.2

In [ ]:
import joblib
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (classification_report, roc_auc_score,
                              average_precision_score, confusion_matrix,
                              recall_score, ConfusionMatrixDisplay)


MODELS_DIR = ML_DIR / 'models'
best_model = joblib.load(MODELS_DIR / 'svm_tuned_model.joblib')
svm_params = joblib.load(MODELS_DIR / 'svm_tuned_params.joblib')

scaler_path = MODELS_DIR / 'svm_scaler.joblib'
if scaler_path.exists():
    scaler = joblib.load(scaler_path)
    print(f'Loaded scaler    : {scaler_path}  (fitted on SMOTE-ENN balanced data)')
else:
    
    print('WARNING: svm_scaler.joblib not found — fitting RobustScaler on X_train as fallback.')
    print('         Re-run ML_Fraud_Detection.ipynb cell 105 to save the correct scaler.')
    scaler = RobustScaler()
    scaler.fit(X_train)

best_thr = svm_params['threshold']

print(f'Loaded model     : {MODELS_DIR / "svm_tuned_model.joblib"}')
print(f'Best C           : {svm_params["C"]:.5f}')
print(f'Decision threshold (from val set): {best_thr:.6f}')

X_test_scaled = scaler.transform(X_test)

y_proba = best_model.predict_proba(X_test_scaled)[:, 1]
y_pred  = (y_proba >= best_thr).astype(int)   

recall_fraud    = recall_score(y_test, y_pred, pos_label=1)
recall_nonfraud = recall_score(y_test, y_pred, pos_label=0)
roc_auc         = roc_auc_score(y_test, y_proba)
avg_prec        = average_precision_score(y_test, y_proba)

print('\nSVM (Tuned) — Test Set Performance')
print('=' * 45)
print(f'Fraud probability  — min={y_proba.min():.4f}, max={y_proba.max():.4f}, mean={y_proba.mean():.4f}')
print(f'ROC-AUC score      : {roc_auc:.4f}')
print(f'Avg Precision (AP) : {avg_prec:.4f}')
print(f'Recall (Fraud=1)   : {recall_fraud:.4f}')
print(f'Recall (Non-fraud) : {recall_nonfraud:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['Non-Fraud', 'Fraud']))

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
print(f'Confusion Matrix (threshold={best_thr:.4f})')
print(f'  TN={tn:>5}  FP={fp:>5}')
print(f'  FN={fn:>5}  TP={tp:>5}')
print()

fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Non-Fraud', 'Fraud'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Confusion Matrix — SVM Tuned (threshold={best_thr:.4f})', fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'confusion_matrix_svm.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: confusion_matrix_svm.png')



## Section 5: SHAP Explainability

In [ ]:
import shap
from sklearn.svm import LinearSVC

if X_train_balanced is not None:
    X_shap_train = scaler.transform(X_train_balanced)
    y_shap_train = y_train_balanced
    print('Using SMOTE-ENN balanced data for SHAP LinearSVC (matches production model).')
else:
    X_shap_train = scaler.transform(X_train)
    y_shap_train = y_train
    print('WARNING: Falling back to raw X_train for SHAP — re-run ML_Fraud_Detection.ipynb cell 105.')

X_train_scaled = X_shap_train

print(f'SHAP training data shape: {X_shap_train.shape}')
print('Fitting LinearSVC for SHAP explanations...')
svm_for_shap = LinearSVC(
    C=svm_params['C'],          
    loss='squared_hinge',
    class_weight='balanced',
    max_iter=5000,
    random_state=42
)
svm_for_shap.fit(X_shap_train, y_shap_train)

print('Computing SHAP values (LinearExplainer)...')
explainer   = shap.LinearExplainer(svm_for_shap, X_shap_train)
shap_values = explainer.shap_values(X_test_scaled)

shap_fraud = shap_values

print(f'SHAP values shape: {shap_fraud.shape}')


In [ ]:
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_fraud, X_test,
    feature_names=feature_names,
    max_display=100,
    show=False
)
plt.title('Global SHAP Feature Importance (Fraud Class)', fontsize=10)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'shap_global_summary.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: shap_global_summary.png')


## Section 6: Identify High-Risk Providers

In [ ]:
def get_top_shap_features(row_idx: int, shap_matrix: np.ndarray,
                           feature_names: List[str], top_n: int = 100) -> List[Dict]:

    shap_row  = shap_matrix[row_idx]
    abs_vals  = np.abs(shap_row)
    top_idx   = np.argsort(abs_vals)[::-1][:top_n]
    
    results = []
    for rank, fi in enumerate(top_idx, start=1):
        results.append({
            'rank': rank,
            'feature': feature_names[fi],
            'shap_value': float(shap_row[fi]),
            'direction': 'increases fraud risk' if shap_row[fi] > 0 else 'decreases fraud risk'
        })
    return results


print(f'Fraud probability range: min={y_proba.min():.6f}, max={y_proba.max():.6f}')
print(f'Decision threshold (from val set, target recall={TARGET_RECALL:.0%}): {best_thr:.6f}')

test_results = X_test.copy()
test_results['true_label']      = y_test.values
test_results['fraud_prob']      = y_proba
test_results['predicted_fraud'] = (y_proba >= best_thr).astype(int)

flagged = test_results[test_results['predicted_fraud'] == 1].copy()
flagged = flagged.sort_values('fraud_prob', ascending=False)

print(f'\nProviders flagged as high-risk: {len(flagged)}')
print(f'True fraudsters among flagged:  {flagged["true_label"].sum()}')
print(f'\nTop 10 highest-risk providers (fraud probability):')
print(flagged[['fraud_prob', 'true_label']].head(10).to_string())



## Section 7: SHAP-to-Language Bridge


In [ ]:

FEATURE_DESCRIPTIONS = {
    'Total_Services_B':         'total number of Medicare Part B services billed',
    'Total_Beneficiaries_B':    'total unique Medicare beneficiaries served (Part B)',
    'Avg_Submitted_Charge_B':   'average submitted charge per Part B service',
    'Max_Submitted_Charge_B':   'maximum submitted charge on a single Part B claim',
    'Avg_Medicare_Allowed_B':   'average Medicare-allowed payment per Part B service',
    'Max_Medicare_Allowed_B':   'highest Medicare-allowed payment on a single Part B service',
    'Weighted_Avg_Payment_B':   'weighted average Medicare payment per Part B service',
    'Unique_HCPCS_B':           'number of distinct HCPCS procedure codes billed (Part B)',
    'Service_Type_Diversity_B': 'variety of service categories billed (Part B)',
    'Unique_Service_Places_B':  'number of distinct places of service where claims were filed',
    'Total_Claims_D':           'total Medicare Part D prescription drug claims',
    'Total_Drug_Cost_D':        'total cost of all Part D drugs prescribed',
    'Avg_Drug_Cost_Per_Claim_D':'average cost per Part D prescription claim',
    'Unique_Brand_Drugs_D':     'number of unique brand-name drugs prescribed (Part D)',
    'Charge_to_Allowed_Ratio':  'ratio of submitted charges to Medicare-allowed amounts (markup ratio)',
    'Payment_to_Allowed_Ratio': 'ratio of actual payment to allowed amount',
    'MaxCharge_Ratio':          'ratio of maximum charge to the average charge',
    'Payment_Efficiency':       'ratio of Medicare payment to submitted charges',
    'Services_per_Beneficiary_B':  'average number of Part B services performed per beneficiary',
    'BeneDay_Per_Beneficiary_B':   'average Medicare beneficiary-days per patient',
    'Services_Per_HCPCS':          'average number of services billed per procedure code',
    'Has_PartD':                   'indicator: provider also prescribes Part D drugs',
    'Has_Both':                    'indicator: provider bills both Part B and Part D',
    'Drug_Cost_Per_Day':           'average daily drug cost per beneficiary',
    'Service_to_Drug_Ratio':       'ratio of Part B services to Part D drug claims',
    'High_Service_Volume_Flag':    'flag: provider is in the top 10% for service volume',
    'High_Drug_Cost_Flag':         'flag: provider is in the top 10% for drug costs',
    'High_Payment_Flag':           'flag: provider is in the top 10% for Medicare payments',
    'High_Service_High_Payment':   'flag: high volume AND high payment (joint risk indicator)',
    'Charge_x_Volume':             'product of charge ratio and service volume (compounded risk)',
    'Z_Total_Services_B':          'standardized z-score for total Part B services vs. peers',
    'Z_Total_Drug_Cost_D':         'standardized z-score for total Part D drug costs vs. peers',
    'Z_Avg_Medicare_Payment_B':    'standardized z-score for average Medicare payment vs. peers',
    'RUCA_Code':                   'Rural-Urban Commuting Area (RUCA) code for provider location',
    'Total_BeneDay_Services_B':    'total Medicare beneficiary-day services (Part B)',
    'log_Total_Services_B':        'log-transformed total Part B services',
    'log_Total_Claims_D':          'log-transformed total Part D claims',
    'Fills_Per_Claim_D':           'average drug fills per Part D claim',
    'Drug_Cost_Per_Claim_D':       'average drug cost per Part D claim',
}

def shap_to_text(top_features: List[Dict]) -> str:
    lines = []
    for f in top_features:
        feat    = f['feature']
        shap_v  = f['shap_value']
        direction = f['direction']
        desc    = FEATURE_DESCRIPTIONS.get(feat, feat.replace('_', ' ').lower())
        strength = 'strongly' if abs(shap_v) > 0.1 else 'moderately'
        lines.append(f"  - {desc} ({feat}): {strength} {direction} [SHAP={shap_v:+.4f}]")
    return '\n'.join(lines)

def build_rag_query_for_provider(top_features: List[Dict]) -> str:
    risk_terms = []
    for f in top_features:
        feat = f['feature']
        if f['shap_value'] > 0:  
            desc = FEATURE_DESCRIPTIONS.get(feat, feat)
            risk_terms.append(desc)
    if not risk_terms:
        return 'Medicare fraud indicators high billing volume unusual claims'
    return 'Medicare fraud investigation: ' + '; '.join(risk_terms[:3])


print('SHAP-to-language bridge defined.')


## Section 8: Claude Report Generation

In [ ]:

REPORT_SYSTEM_PROMPT = """You are a Medicare fraud investigation analyst working for the
Office of Inspector General (OIG). You produce structured, evidence-based investigation
reports that help investigators decide whether to escalate a case.

Your reports must:
- Be professional, precise, and grounded in the provided policy excerpts
- Cite specific Medicare manual sections or OIG guidance wherever relevant, including
  page numbers, chapter headings, and subsection references when they are available
- Distinguish between definitive fraud indicators and patterns that require further investigation
- Avoid speculation — flag uncertainty where evidence is incomplete
"""

def _format_policy_citation(chunk: Dict) -> str:
    
    parts = [f"[Source: {chunk['source_category']} | {chunk['source_file']}"]

    sp, ep = chunk.get('start_page'), chunk.get('end_page')
    if sp and ep and sp != ep:
        parts.append(f"pp. {sp}–{ep}")
    elif sp:
        parts.append(f"p. {sp}")

    if chunk.get('chapter'):
        parts.append(f"Chapter: {chunk['chapter'][:60]}")
    if chunk.get('subsection'):
        parts.append(f"§ {chunk['subsection'][:60]}")

    return ' | '.join(parts) + ']'


def build_report_prompt(
    provider_id: int,
    fraud_probability: float,
    true_label: int,
    shap_narrative: str,
    policy_chunks: List[Dict]
) -> str:
    
    label_str = '(CONFIRMED FRAUDSTER)' if true_label == 1 else '(model-flagged, unconfirmed)'

    policy_context = '\n\n'.join([
        f"{_format_policy_citation(c)}\n{c['text'][:400]}"
        for c in policy_chunks[:4]
    ])

    prompt = f"""Generate a structured Medicare fraud investigation report for the provider below.

PROVIDER SUMMARY
  Provider ID      : {provider_id}
  Fraud Probability: {fraud_probability:.1%}  {label_str}

MACHINE LEARNING — TOP RISK FACTORS (SHAP analysis)
{shap_narrative}

RELEVANT POLICY EXCERPTS (retrieved from Medicare manuals and OIG guidance)
{policy_context}

Please produce a report with these sections:
1. EXECUTIVE SUMMARY (3–4 sentences: risk level, key drivers, recommended action)
2. FRAUD RISK FACTORS (analyze each SHAP factor in light of Medicare policy)
3. REGULATORY CONTEXT (which Medicare rules or OIG guidelines are potentially violated,
   citing manual name, chapter, subsection, and page number where available)
4. INVESTIGATION RECOMMENDATIONS (specific next steps for investigators)
5. CONFIDENCE ASSESSMENT (how confident is this flag, and what would confirm/dismiss it)
"""
    return prompt



def generate_provider_report(
    provider_id: int,
    fraud_probability: float,
    true_label: int,
    shap_narrative: str,
    policy_chunks: List[Dict],
    model: str = CLAUDE_MODEL
) -> str:
    """Call Claude to generate a fraud investigation report."""
    prompt = build_report_prompt(
        provider_id, fraud_probability, true_label, shap_narrative, policy_chunks
    )
    response = client.messages.create(
        model=model,
        system=REPORT_SYSTEM_PROMPT,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.2,
        max_tokens=1200
    )
    return response.content[0].text


print('Report generation functions defined.')



## Section 9: Generate Reports for All Flagged Providers

In [ ]:
providers_to_process = flagged.head(MAX_PROVIDERS_TO_REPORT)
print(f'Generating reports for {len(providers_to_process)} providers...')
print(f'(Capped at MAX_PROVIDERS_TO_REPORT={MAX_PROVIDERS_TO_REPORT} to manage API cost)')

report_records = []

for i, (provider_idx, provider_row) in enumerate(
    tqdm(providers_to_process.iterrows(), total=len(providers_to_process), desc='Generating reports')
):
    
    test_pos = X_test.index.get_loc(provider_idx)

    top_features   = get_top_shap_features(test_pos, shap_fraud, feature_names, TOP_SHAP_FEATURES)
    shap_narrative = shap_to_text(top_features)

    rag_query     = build_rag_query_for_provider(top_features)
    policy_chunks = retrieve(rag_query, rag_index, chunk_texts, chunk_metadata, top_k=5)

    try:
        report_text = generate_provider_report(
            provider_id=provider_idx,
            fraud_probability=provider_row['fraud_prob'],
            true_label=int(provider_row['true_label']),
            shap_narrative=shap_narrative,
            policy_chunks=policy_chunks
        )
    except Exception as e:
        report_text = f'Error generating report: {e}'

    report_path = REPORTS_DIR / f'provider_{provider_idx}_report.txt'
    with open(report_path, 'w') as f:
        f.write(f'MEDICARE FRAUD INVESTIGATION REPORT\n')
        f.write(f'Provider ID: {provider_idx}\n')
        f.write(f'Fraud Probability: {provider_row["fraud_prob"]:.1%}\n')
        f.write(f'True Label: {int(provider_row["true_label"])}\n')
        f.write(f'Model: SVM (Tuned) — LinearSVC C={svm_params["C"]:.5f}, CalibratedClassifierCV\n')
        f.write(f'Detection Threshold: {best_thr:.6f} (target recall={TARGET_RECALL:.0%})\n')
        f.write(f'\n--- SHAP ANALYSIS ---\n{shap_narrative}\n')
        f.write(f'\n--- RAG POLICY SOURCES ---\n')
        for pc in policy_chunks[:3]:
            sp, ep = pc.get('start_page'), pc.get('end_page')
            if sp and ep and sp != ep:
                page_str = f'pp. {sp}–{ep}'
            elif sp:
                page_str = f'p. {sp}'
            else:
                page_str = 'page unknown'
            chapter_str    = pc.get('chapter')    or 'n/a'
            subsection_str = pc.get('subsection') or 'n/a'
            chunk_id_str   = pc.get('chunk_id')   or 'n/a'
            f.write(f"  [{pc['source_category']}] {pc['source_file']}\n")
            f.write(f"    Pages: {page_str} | Chapter: {chapter_str[:60]} | §: {subsection_str[:60]}\n")
            f.write(f"    Chunk ID: {chunk_id_str}\n")
        f.write(f'\n--- CLAUDE INVESTIGATION REPORT ---\n{report_text}\n')

    report_records.append({
        'provider_id':          provider_idx,
        'fraud_probability':    provider_row['fraud_prob'],
        'true_label':           int(provider_row['true_label']),
        'detection_threshold':  best_thr,
        'top_feature_1':        top_features[0]['feature'] if len(top_features) > 0 else '',
        'top_feature_2':        top_features[1]['feature'] if len(top_features) > 1 else '',
        'top_feature_3':        top_features[2]['feature'] if len(top_features) > 2 else '',
        'rag_query':            rag_query,
        'policy_source_1':      policy_chunks[0]['source_category'] if policy_chunks else '',
        'policy_page_1':        policy_chunks[0].get('start_page')  if policy_chunks else None,
        'policy_chapter_1':     (policy_chunks[0].get('chapter')    or '')[:80] if policy_chunks else '',
        'policy_subsection_1':  (policy_chunks[0].get('subsection') or '')[:80] if policy_chunks else '',
        'policy_chunk_id_1':    (policy_chunks[0].get('chunk_id')   or '')      if policy_chunks else '',
        'report_path':          str(report_path)
    })

report_df = pd.DataFrame(report_records)
report_df.to_csv(REPORTS_DIR / 'provider_reports_summary.csv', index=False)
print(f'\nAll reports saved to: {REPORTS_DIR}')
print(f'Summary CSV: {REPORTS_DIR / "provider_reports_summary.csv"}')



## Section 10: Preview a Sample Report

In [ ]:
if report_records:
    sample = report_records[0]
    print(f'Provider ID: {sample["provider_id"]}')
    print(f'Fraud Probability: {sample["fraud_probability"]:.1%}')
    print(f'True Label: {sample["true_label"]} (1=confirmed fraud, 0=non-fraud)')
    print(f'\n--- Report ---')
    with open(sample['report_path'], 'r') as f:
        print(f.read())


## Section 11: SHAP Waterfall Plot for Top Flagged Provider

In [ ]:
if len(flagged) > 0:
    top_provider_idx = flagged.index[0]
    test_pos = X_test.index.get_loc(top_provider_idx)

    shap_explanation = shap.Explanation(
        values=shap_fraud[test_pos],
        base_values=explainer.expected_value[1] if isinstance(explainer.expected_value, (list, np.ndarray)) else explainer.expected_value,
        data=X_test_scaled[test_pos],
        feature_names=feature_names
    )

    shap.waterfall_plot(shap_explanation, max_display=12, show=False)
    plt.title(f'SHAP Waterfall — Provider {top_provider_idx}\n'
              f'(Fraud Prob = {flagged.loc[top_provider_idx, "fraud_prob"]:.1%})', fontsize=13)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f'shap_waterfall_provider_{top_provider_idx}.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Waterfall plot saved.')


## Section 12: Summary Statistics

In [ ]:
print('=' * 55)
print('PROVIDER REPORT GENERATION — COMPLETE')
print('=' * 55)
print(f'Providers flagged    : {len(flagged)}')
print(f'Reports generated    : {len(report_records)}')
print(f'True fraud (flagged) : {report_df["true_label"].sum()}')
print(f'\nOutput directory: {REPORTS_DIR}')
print(f'\nFiles produced:')
for f in sorted(REPORTS_DIR.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:<50} {size_kb:.1f} KB')
print(f'\nNext step: Open LLM_3_Fraud_Investigator_Chatbot.ipynb')